# Initial Value Problem Solvers

This notebook demonstrates `torchscience.integration.initial_value_problem` - differentiable ODE solvers for PyTorch.

We'll explore three physics examples:
1. **Double Pendulum** - chaotic dynamics with adaptive stepping
2. **Lotka-Volterra** - ecology model with TensorDict state
3. **Neural ODE** - learning dynamics with adjoint gradients

Each example showcases different solver capabilities with interactive Plotly visualizations and Manim animations.

In [ ]:
import time

import plotly.express as px  # noqa: F401
import plotly.graph_objects as go  # noqa: F401
import torch  # noqa: F401
from plotly.subplots import make_subplots  # noqa: F401
from tensordict import TensorDict  # noqa: F401

try:
    from manim import *  # noqa: F401, F403

    MANIM_AVAILABLE = True
except ImportError:
    MANIM_AVAILABLE = False
    print("Note: manim not installed. Animation examples will be skipped.")
from IPython.display import Video  # noqa: F401

from torchscience.integration.initial_value_problem import (  # noqa: F401
    adjoint,
    backward_euler,
    dormand_prince_5,
    euler,
    midpoint,
    runge_kutta_4,
)

In [ ]:
# Configuration
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Plotly defaults for dark theme
plotly_template = "plotly_dark"  # noqa: F841

## 1. Double Pendulum: Chaos and Adaptive Stepping

The double pendulum is a classic chaotic system - two rigid rods connected at joints, swinging under gravity. Small changes in initial conditions lead to dramatically different trajectories.

**State vector:** `[theta1, theta2, omega1, omega2]`
- `theta1`, `theta2`: angles from vertical
- `omega1`, `omega2`: angular velocities

**Why it's a good test case:**
- Chaotic dynamics require adaptive step sizes
- Large state changes stress error estimation
- Visually dramatic for demonstrations

In [ ]:
def double_pendulum_dynamics(t, state):
    """
    Double pendulum equations of motion.

    Parameters:
        t: time (unused, system is autonomous)
        state: [theta1, theta2, omega1, omega2]

    Returns:
        [dtheta1/dt, dtheta2/dt, domega1/dt, domega2/dt]
    """
    # Physical parameters
    g = 9.81  # gravity
    L1 = 1.0  # length of rod 1
    L2 = 1.0  # length of rod 2
    m1 = 1.0  # mass 1
    m2 = 1.0  # mass 2

    theta1, theta2, omega1, omega2 = (
        state[..., 0],
        state[..., 1],
        state[..., 2],
        state[..., 3],
    )

    delta = theta2 - theta1

    # Denominator terms
    den1 = (m1 + m2) * L1 - m2 * L1 * torch.cos(delta) ** 2
    den2 = (L2 / L1) * den1

    # Angular accelerations
    domega1 = (
        -m2 * L1 * omega1**2 * torch.sin(delta) * torch.cos(delta)
        + m2 * g * torch.sin(theta2) * torch.cos(delta)
        + m2 * L2 * omega2**2 * torch.sin(delta)
        - (m1 + m2) * g * torch.sin(theta1)
    ) / den1

    domega2 = (
        +m2 * L2 * omega2**2 * torch.sin(delta) * torch.cos(delta)
        + (m1 + m2) * g * torch.sin(theta1) * torch.cos(delta)
        + (m1 + m2) * L1 * omega1**2 * torch.sin(delta)
        - (m1 + m2) * g * torch.sin(theta2)
    ) / den2

    return torch.stack([omega1, omega2, domega1, domega2], dim=-1)

In [ ]:
# Initial conditions: starting nearly inverted for dramatic chaos
# [theta1, theta2, omega1, omega2]
y0 = torch.tensor([3.0, 3.0, 0.0, 0.0], dtype=torch.float64, device=device)

# Solve with dormand_prince_5
start = time.perf_counter()
y_final, interp = dormand_prince_5(
    double_pendulum_dynamics,
    y0,
    t_span=(0.0, 20.0),
    rtol=1e-8,
    atol=1e-10,
)
elapsed = time.perf_counter() - start

print(f"Integration completed in {elapsed:.3f}s")
print(f"Number of steps: {interp.n_steps}")
print(f"Final state: theta1={y_final[0]:.3f}, theta2={y_final[1]:.3f}")